In [3]:
!pip install biopython

In [4]:
from Bio import Entrez, SeqIO

Entrez.email = "ysb070828@gmail.com"

accessions = [
    "NP_000184.1",
    "NP_033196.1",
    "NP_990152.1",
    "XP_003221976.1",
    "XP_007433256.1",
    "XP_015688035.1"
]

with open("shh_selected.fasta", "w") as out_handle:
    for acc in accessions:
        handle = Entrez.efetch(db="protein", id=acc, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        handle.close()
        SeqIO.write(record, out_handle, "fasta")
        print(record.id, record.description)

NP_000184.1 NP_000184.1 sonic hedgehog protein isoform 1 preproprotein [Homo sapiens]
NP_033196.1 NP_033196.1 sonic hedgehog protein isoform 1 precursor [Mus musculus]
NP_990152.1 NP_990152.1 sonic hedgehog protein precursor [Gallus gallus]
XP_003221976.1 XP_003221976.1 sonic hedgehog protein [Anolis carolinensis]
XP_007433256.1 XP_007433256.1 sonic hedgehog protein [Python bivittatus]
XP_015688035.1 XP_015688035.1 sonic hedgehog protein isoform X1 [Protobothrops mucrosquamatus]


In [12]:
from Bio import SeqIO
import re

records = list(SeqIO.parse("shh_selected.fasta", "fasta"))

for record in records:
    match = re.search(r"\[(.*?)\]", record.description)
    if match:
        species = match.group(1)
    else:
        species = "Unknown"

    print(species)
    print("length:", len(record.seq))
    print(record.seq[:20], "...")
    print("-" * 40)

Homo sapiens
length: 462
MLLLARCLLLVLVSSLLVCS ...
----------------------------------------
Mus musculus
length: 437
MLLLLARCFLVILASSLLVC ...
----------------------------------------
Gallus gallus
length: 425
MVEMLLLTRILLVGFICALL ...
----------------------------------------
Anolis carolinensis
length: 427
MRVPWGAGRWCRAPALLLLL ...
----------------------------------------
Python bivittatus
length: 426
MLLRRRSGLLPLCLGALFLS ...
----------------------------------------
Protobothrops mucrosquamatus
length: 426
MLLLRRTGLLPLCLGALFLS ...
----------------------------------------


In [13]:
from Bio import SeqIO
import re

records = list(SeqIO.parse("shh_selected.fasta", "fasta"))

for record in records:
    match = re.search(r"\[(.*?)\]", record.description)
    
    if match:
        species = match.group(1)
    else:
        species = "Unknown"

    print(f"{species}\t{len(record.seq)} aa")

Homo sapiens	462 aa
Mus musculus	437 aa
Gallus gallus	425 aa
Anolis carolinensis	427 aa
Python bivittatus	426 aa
Protobothrops mucrosquamatus	426 aa


In [19]:
from Bio import SeqIO, pairwise2
import re

records = list(SeqIO.parse("shh_selected.fasta", "fasta"))

def get_species(record):
    match = re.search(r"\[(.*?)\]", record.description)
    if match:
        return match.group(1)
    else:
        return "Unknown"

print(f"{'Species1':<30}{'Species2':<30}{'Similarity (%)':>15}")

for i in range(len(records)):
    for j in range(i+1, len(records)):
        seq1 = str(records[i].seq)
        seq2 = str(records[j].seq)

        alignment = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True)[0]
        score = alignment.score
        max_len = max(len(seq1), len(seq2))
        similarity = score / max_len * 100

        sp1 = get_species(records[i])
        sp2 = get_species(records[j])

        print(f"{sp1:<30}{sp2:<30}{similarity:>15.2f}")

Species1                      Species2                       Similarity (%)
Homo sapiens                  Mus musculus                            88.74
Homo sapiens                  Gallus gallus                           79.65
Homo sapiens                  Anolis carolinensis                     77.92
Homo sapiens                  Python bivittatus                       76.41
Homo sapiens                  Protobothrops mucrosquamatus            75.97
Mus musculus                  Gallus gallus                           83.30
Mus musculus                  Anolis carolinensis                     81.46
Mus musculus                  Python bivittatus                       80.78
Mus musculus                  Protobothrops mucrosquamatus            79.63
Gallus gallus                 Anolis carolinensis                     80.80
Gallus gallus                 Python bivittatus                       81.92
Gallus gallus                 Protobothrops mucrosquamatus            81.46
Anolis carol

In [22]:
import requests
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import csv

ENSEMBL = "https://rest.ensembl.org"

# human ZRS core region (hg38)
REF_SPECIES = "homo_sapiens"
REF_REGION = "7:156791102-156791874"

TARGET_SPECIES = [
    "homo_sapiens",
    "mus_musculus",
    "gallus_gallus",
    "anolis_carolinensis",
    "pseudonaja_textilis"
]

OUTDIR = Path("zrs_5species_results")
OUTDIR.mkdir(exist_ok=True)

# ======================================
# 2. 유틸 함수
# ======================================

def get_json(url, params=None):
    r = requests.get(
        url,
        params=params,
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        timeout=120,
    )
    if not r.ok:
        print("Request URL:", r.url)
        print("Status code:", r.status_code)
        print("Response text:", r.text[:1000])
        r.raise_for_status()
    return r.json()

def wrap_fasta(seq, width=60):
    return "\n".join(seq[i:i+width] for i in range(0, len(seq), width))

def normalize_species_name(name):
    return name.strip().lower()

def strip_gaps(seq):
    return seq.replace("-", "")

def pairwise_stats(seq1, seq2):
    """
    seq1, seq2: already aligned strings of the same length
    """
    if len(seq1) != len(seq2):
        raise ValueError("Aligned sequences must have the same length.")

    matches = 0
    mismatches = 0
    gaps = 0
    comparable = 0  # both are not '-'

    for a, b in zip(seq1, seq2):
        if a == "-" or b == "-":
            gaps += 1
        else:
            comparable += 1
            if a == b:
                matches += 1
            else:
                mismatches += 1

    identity_alignment = matches / len(seq1) * 100 if seq1 else 0.0
    identity_ungapped = matches / comparable * 100 if comparable else 0.0

    return {
        "alignment_length": len(seq1),
        "matches": matches,
        "mismatches": mismatches,
        "gaps": gaps,
        "comparable_sites": comparable,
        "identity_alignment_%": round(identity_alignment, 2),
        "identity_ungapped_%": round(identity_ungapped, 2),
    }

# ======================================
# 3. Ensembl species 확인
# ======================================

print("=== Ensembl species 확인 ===")
species_info = get_json(f"{ENSEMBL}/info/species")
available_species = {normalize_species_name(x["name"]) for x in species_info["species"]}

for sp in TARGET_SPECIES:
    if sp in available_species:
        print(f"[OK] {sp}")
    else:
        print(f"[NOT FOUND] {sp}")

# ======================================
# 4. ZRS 다중 정렬 다운로드
# ======================================
# amniotes group 사용
# method=PECAN 시도 -> 실패하면 method 없이 재시도
# (Ensembl 서버 버전에 따라 조합이 다를 수 있음)

print("\n=== ZRS alignment download ===")

def fetch_alignment_blocks():
    trials = [
        {"species_set_group": "amniotes", "method": "PECAN", "aligned": 1},
        {"species_set_group": "amniotes", "aligned": 1},
    ]

    last_error = None
    for params in trials:
        try:
            print("Trying params:", params)
            blocks = get_json(
                f"{ENSEMBL}/alignment/region/{REF_SPECIES}/{REF_REGION}",
                params=params
            )
            if blocks:
                return blocks, params
        except Exception as e:
            print("Failed with:", params)
            print("Error:", e)
            last_error = e

    raise RuntimeError(f"정렬 블록 다운로드 실패: {last_error}")

blocks, used_params = fetch_alignment_blocks()
print("Used params:", used_params)
print("Number of blocks:", len(blocks))

# ======================================
# 5. 블록에서 종별 aligned sequence 이어붙이기
# ======================================

species_to_seq_parts = defaultdict(list)

for block in blocks:
    for aln in block.get("alignments", []):
        sp = normalize_species_name(aln["species"])
        seq = aln.get("seq", "")
        if sp in TARGET_SPECIES:
            species_to_seq_parts[sp].append(seq)

species_to_aligned = {}
for sp in TARGET_SPECIES:
    parts = species_to_seq_parts.get(sp, [])
    if parts:
        species_to_aligned[sp] = "".join(parts)

print("\n=== Retrieved species ===")
for sp in TARGET_SPECIES:
    if sp in species_to_aligned:
        print(f"[DOWNLOADED] {sp}\taligned_len={len(species_to_aligned[sp])}")
    else:
        print(f"[MISSING]     {sp}")

if REF_SPECIES not in species_to_aligned:
    raise RuntimeError("human reference sequence 내려받지 못함")

# ======================================
# 6. FASTA 저장
# ======================================

aligned_fasta = OUTDIR / "zrs_5species_aligned.fasta"
with open(aligned_fasta, "w", encoding="utf-8") as f:
    for sp in TARGET_SPECIES:
        seq = species_to_aligned.get(sp)
        if seq:
            f.write(f">{sp}\n{wrap_fasta(seq)}\n")

ungapped_fasta = OUTDIR / "zrs_5species_ungapped.fasta"
with open(ungapped_fasta, "w", encoding="utf-8") as f:
    for sp in TARGET_SPECIES:
        seq = species_to_aligned.get(sp)
        if seq:
            f.write(f">{sp}\n{wrap_fasta(strip_gaps(seq))}\n")

print("\nSaved:")
print("-", aligned_fasta)
print("-", ungapped_fasta)

# ======================================
# 7. 종별 길이 요약
# ======================================

length_csv = OUTDIR / "zrs_sequence_lengths.csv"
with open(length_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["species", "aligned_length", "ungapped_length"])
    for sp in TARGET_SPECIES:
        seq = species_to_aligned.get(sp)
        if seq:
            writer.writerow([sp, len(seq), len(strip_gaps(seq))])

print("-", length_csv)

# ======================================
# 8. pairwise 비교 통계
# ======================================

stats_csv = OUTDIR / "zrs_pairwise_stats.csv"
rows = []

available = [sp for sp in TARGET_SPECIES if sp in species_to_aligned]

for sp1, sp2 in combinations(available, 2):
    seq1 = species_to_aligned[sp1]
    seq2 = species_to_aligned[sp2]

    # block 이어붙인 결과라 길이는 같아야 정상
    if len(seq1) != len(seq2):
        print(f"[WARN] Length mismatch: {sp1} vs {sp2}")
        continue

    s = pairwise_stats(seq1, seq2)
    row = {"species1": sp1, "species2": sp2}
    row.update(s)
    rows.append(row)

with open(stats_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "species1", "species2",
            "alignment_length",
            "matches", "mismatches", "gaps", "comparable_sites",
            "identity_alignment_%", "identity_ungapped_%"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

print("-", stats_csv)

# ======================================
# 9. human 기준 비교 결과 출력
# ======================================

print("\n=== Human 기준 요약 ===")
for row in rows:
    if row["species1"] == "homo_sapiens" or row["species2"] == "homo_sapiens":
        other = row["species2"] if row["species1"] == "homo_sapiens" else row["species1"]
        print(
            f"human vs {other}: "
            f"identity_ungapped={row['identity_ungapped_%']}%, "
            f"mismatches={row['mismatches']}, gaps={row['gaps']}"
        )

print("\n완료.")

=== Ensembl species 확인 ===
[OK] homo_sapiens
[OK] mus_musculus
[OK] gallus_gallus
[OK] anolis_carolinensis
[OK] pseudonaja_textilis

=== ZRS alignment download ===
Trying params: {'species_set_group': 'amniotes', 'method': 'PECAN', 'aligned': 1}
Used params: {'species_set_group': 'amniotes', 'method': 'PECAN', 'aligned': 1}
Number of blocks: 1

=== Retrieved species ===
[DOWNLOADED] homo_sapiens	aligned_len=1392
[DOWNLOADED] mus_musculus	aligned_len=1392
[DOWNLOADED] gallus_gallus	aligned_len=1392
[DOWNLOADED] anolis_carolinensis	aligned_len=1392
[DOWNLOADED] pseudonaja_textilis	aligned_len=1392

Saved:
- zrs_5species_results\zrs_5species_aligned.fasta
- zrs_5species_results\zrs_5species_ungapped.fasta
- zrs_5species_results\zrs_sequence_lengths.csv
- zrs_5species_results\zrs_pairwise_stats.csv

=== Human 기준 요약 ===
human vs mus_musculus: identity_ungapped=89.37%, mismatches=81, gaps=630
human vs gallus_gallus: identity_ungapped=88.17%, mismatches=91, gaps=623
human vs anolis_carolinens

In [23]:
import re
import csv
from pathlib import Path
from Bio import SeqIO

aligned_fasta = Path("zrs_5species_results/zrs_5species_aligned.fasta")
outdir = Path("zrs_5species_results")
outdir.mkdir(exist_ok=True)

species_order = [
    "homo_sapiens",
    "mus_musculus",
    "gallus_gallus",
    "anolis_carolinensis",
    "pseudonaja_textilis"
]


ETS_PATTERNS = {
    "ETS_core_GGAA": r"GGAA",
    "ETS_core_GGAT": r"GGAT",
}

FLANK = 8


# ======================================
# 2) FASTA 읽기
# ======================================

records = {rec.id: str(rec.seq).upper() for rec in SeqIO.parse(aligned_fasta, "fasta")}

for sp in species_order:
    if sp not in records:
        raise ValueError(f"{sp} 가 FASTA에 없음")

human_aligned = records["homo_sapiens"]

aligned_lengths = {sp: len(records[sp]) for sp in species_order}
if len(set(aligned_lengths.values())) != 1:
    raise ValueError(f"Aligned length mismatch: {aligned_lengths}")

aligned_len = len(human_aligned)


# ======================================
# 3) aligned <-> ungapped 좌표 변환용 함수
# ======================================

def build_aligned_to_ungapped_map(aligned_seq):
    """
    aligned index i -> ungapped position (1-based), gap이면 None
    """
    mapping = []
    pos = 0
    for ch in aligned_seq:
        if ch == "-":
            mapping.append(None)
        else:
            pos += 1
            mapping.append(pos)
    return mapping

def build_ungapped_to_aligned_map(aligned_seq):
    """
    ungapped position (1-based) -> aligned index (0-based)
    """
    mapping = {}
    pos = 0
    for i, ch in enumerate(aligned_seq):
        if ch != "-":
            pos += 1
            mapping[pos] = i
    return mapping

human_a2u = build_aligned_to_ungapped_map(human_aligned)
human_u2a = build_ungapped_to_aligned_map(human_aligned)
human_ungapped = human_aligned.replace("-", "")


# ======================================
# 4) motif 검색
# ======================================

def reverse_complement(seq):
    comp = str.maketrans("ACGT", "TGCA")
    return seq.translate(comp)[::-1]

def find_motifs_in_human_ungapped(seq, pattern_name, regex):
    """
    human ungapped sequence에서 motif를 찾고,
    ungapped 좌표 + aligned 좌표로 반환
    """
    hits = []
    for m in re.finditer(regex, seq):
        start_u = m.start() + 1   
        end_u = m.end()           
        motif_seq = m.group()

        # ungapped -> aligned
        start_a = human_u2a[start_u]
        end_a = human_u2a[end_u]

        hits.append({
            "pattern": pattern_name,
            "motif_seq_human": motif_seq,
            "strand": "+",
            "start_ungapped_human": start_u,
            "end_ungapped_human": end_u,
            "start_aligned": start_a,
            "end_aligned": end_a,
        })

    rc = reverse_complement(regex.replace("[AT]", "W"))
    return hits

all_hits = []
for pattern_name, regex in ETS_PATTERNS.items():
    all_hits.extend(find_motifs_in_human_ungapped(human_ungapped, pattern_name, regex))

all_hits.sort(key=lambda x: (x["start_ungapped_human"], x["pattern"]))


# ======================================
# 5) 정렬 위치 기준으로 각 종 motif 상태 평가
# ======================================

def classify_site(human_motif, other_seq):
    """
    human motif와 같은 aligned column 구간에서
    다른 종의 motif 상태를 평가
    """
    if "-" in other_seq:
        return "gap_disrupted"
    elif other_seq == human_motif:
        return "conserved"
    else:
        return "mutated"

def safe_slice(seq, start, end):
    start = max(0, start)
    end = min(len(seq), end)
    return seq[start:end]

rows = []

for idx, hit in enumerate(all_hits, start=1):
    a_start = hit["start_aligned"]
    a_end = hit["end_aligned"]   # inclusive 0-based
    motif_len = a_end - a_start + 1

    human_site = human_aligned[a_start:a_end+1]

    flank_start = max(0, a_start - FLANK)
    flank_end = min(aligned_len, a_end + FLANK + 1)

    row = {
        "site_id": f"ETS_{idx}",
        "pattern": hit["pattern"],
        "human_motif": human_site,
        "human_start_ungapped": hit["start_ungapped_human"],
        "human_end_ungapped": hit["end_ungapped_human"],
        "aligned_start_1based": a_start + 1,
        "aligned_end_1based": a_end + 1,
        "human_context": human_aligned[flank_start:flank_end],
    }

    for sp in species_order:
        site_seq = records[sp][a_start:a_end+1]
        context_seq = records[sp][flank_start:flank_end]

        row[f"{sp}_site"] = site_seq
        row[f"{sp}_context"] = context_seq

        if sp == "homo_sapiens":
            row[f"{sp}_status"] = "reference"
        else:
            row[f"{sp}_status"] = classify_site(human_site, site_seq)

    rows.append(row)


# ======================================
# 6) 저장
# ======================================

out_csv = outdir / "zrs_ETS_motif_scan.csv"
fieldnames = [
    "site_id", "pattern", "human_motif",
    "human_start_ungapped", "human_end_ungapped",
    "aligned_start_1based", "aligned_end_1based",
    "human_context"
]

for sp in species_order:
    fieldnames.extend([
        f"{sp}_site",
        f"{sp}_status",
        f"{sp}_context",
    ])

with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved: {out_csv}")


# ======================================
# 7) snake에서 깨진 ETS 후보만 별도 요약 출력
# ======================================

snake = "pseudonaja_textilis"

print("\n=== Snake에서 훼손된 ETS motif 후보 ===")
count_disrupted = 0

for row in rows:
    status = row[f"{snake}_status"]
    if status != "conserved":
        count_disrupted += 1
        print(
            f"{row['site_id']} | {row['pattern']} | "
            f"human={row['human_motif']} | "
            f"snake={row[f'{snake}_site']} | "
            f"status={status} | "
            f"human_pos={row['human_start_ungapped']}-{row['human_end_ungapped']}"
        )

print(f"\n총 {count_disrupted}개 후보가 snake에서 보존되지 않았음.")


# ======================================
# 8) 종별 보존/변이/gap 개수 요약
# ======================================

summary_csv = outdir / "zrs_ETS_motif_summary.csv"

summary_rows = []
for sp in species_order:
    if sp == "homo_sapiens":
        continue

    conserved = sum(1 for r in rows if r[f"{sp}_status"] == "conserved")
    mutated = sum(1 for r in rows if r[f"{sp}_status"] == "mutated")
    gap_disrupted = sum(1 for r in rows if r[f"{sp}_status"] == "gap_disrupted")

    summary_rows.append({
        "species": sp,
        "conserved": conserved,
        "mutated": mutated,
        "gap_disrupted": gap_disrupted,
        "total_sites": len(rows),
    })

with open(summary_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["species", "conserved", "mutated", "gap_disrupted", "total_sites"]
    )
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"Saved: {summary_csv}")

Saved: zrs_5species_results\zrs_ETS_motif_scan.csv

=== Snake에서 훼손된 ETS motif 후보 ===
ETS_2 | ETS_core_GGAA | human=GG-A----A | snake=GA-T----T | status=gap_disrupted | human_pos=345-348
ETS_3 | ETS_core_GGAT | human=GGAT | snake=GGAA | status=mutated | human_pos=442-445
ETS_4 | ETS_core_GGAA | human=GGAA | snake=---- | status=gap_disrupted | human_pos=572-575
ETS_5 | ETS_core_GGAT | human=GGAT | snake=---- | status=gap_disrupted | human_pos=685-688
ETS_6 | ETS_core_GGAT | human=GGAT | snake=---- | status=gap_disrupted | human_pos=693-696
ETS_7 | ETS_core_GGAT | human=GGAT | snake=---- | status=gap_disrupted | human_pos=702-705
ETS_8 | ETS_core_GGAT | human=GGAT | snake=---- | status=gap_disrupted | human_pos=715-718

총 7개 후보가 snake에서 보존되지 않았음.
Saved: zrs_5species_results\zrs_ETS_motif_summary.csv
